# 실습 1. LlamaIndex — Index 종류와 QueryEngine

## 실습 목표

LlamaIndex는 **데이터 인덱싱·검색에 특화**된 프레임워크입니다(교안 1장). 핵심은 **인덱스가
"무엇을 읽는가"가 종류마다 다르다**는 점 — 검색형(VectorStore)·전체형(Summary)·계층형(Tree).

- `VectorStoreIndex` : 질의와 유사한 **top-k 노드만** 검색 → **QA**에 적합
- `SummaryIndex` : **모든 노드**를 보유 → `tree_summarize`로 **"전체 문서 요약"** 에 적합
- `TreeIndex` : 노드를 **계층 트리**로 요약·구성 → 큰 코퍼스의 하향 탐색
- 같은 한국어 코퍼스(day21 SAMPLE_DOCS)·같은 질의로 6교시 비교에 사용

> LLM 호출이 있으므로 루트 `.env`의 `MLAPI_*`가 필요합니다.

## 1. 공통 설정 + VectorStoreIndex (교안 1.3)

임베딩·LLM을 전역 `Settings`로 지정하고, `from_documents` 한 줄로 청킹·임베딩·인덱싱.

In [ ]:
!pip install llama-index llama-index-embeddings-huggingface llama-index-llms-openai-like sentence-transformers

In [3]:
from _common import (SAMPLE_DOCS, EVAL_QUESTION, MLAPI_BASE_URL, MLAPI_API_KEY,
                     MLAPI_MODEL, EMB_MODEL, have_mlapi)
from llama_index.core import (VectorStoreIndex, SummaryIndex, TreeIndex, Document, Settings)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike

Settings.embed_model = HuggingFaceEmbedding(model_name=EMB_MODEL)
Settings.llm = OpenAILike(model=MLAPI_MODEL, api_base=MLAPI_BASE_URL, api_key=MLAPI_API_KEY,
                          is_chat_model=True, context_window=8192, temperature=0.2)
DOCS = [Document(text=d["text"], metadata={"topic": d["topic"]}) for d in SAMPLE_DOCS]

def build_query_engine(top_k=3):
    index = VectorStoreIndex.from_documents(DOCS)             # 한 줄로 인덱싱
    return index.as_query_engine(similarity_top_k=top_k)

print(f"준비 완료 · 문서 {len(DOCS)}개")

/Users/bkk/프로젝트/yeardream/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11162.22it/s]


준비 완료 · 문서 10개


## 2. VectorStoreIndex 질의 — 검색형 QA (교안 1.3 실측)

질의와 유사한 **top-k 노드만** 검색해 답한다. `resp.source_nodes` 로 근거 출처를 추적.

In [4]:
qe = build_query_engine(top_k=3)
resp = qe.query(EVAL_QUESTION)
print(f"질문: {EVAL_QUESTION}\n")
print(f"답변: {str(resp).strip()[:200]}\n")
print(f"출처(source_nodes) {len(resp.source_nodes)}개:",
      [n.metadata.get("topic","?") for n in resp.source_nodes])

질문: RAG와 파인튜닝의 차이는?

답변: RAG와 파인튜닝의 핵심 차이는 **지식을 다루는 방식**입니다.

- **파인튜닝**
  - 사전학습된 모델의 가중치를 추가 학습시키는 방식
  - **말투, 형식, 특정 작업 성능**을 높이는 데 강함
  - **행동·스타일을 고정**하고 싶을 때 유리
  - 반면 **최신 사실을 자주 반영**하는 데는 적합하지 않음

- **RAG**
  - 답변 전

출처(source_nodes) 3개: ['파인튜닝', 'RAG', '평가']


**포인트 — VectorStoreIndex = 검색형**

- `from_documents` 한 줄로 청킹·임베딩·인덱싱 → '인덱스 추상화'가 LlamaIndex의 개발 편의성
- 질의와 **유사한 top-k**만 본다 → 특정 질문에 답하는 **QA**에 최적
- 단, "전체를 요약해줘" 같은 질의엔 **top-k만 보면 일부 문서가 빠진다** → 3·4절의 SummaryIndex

## 3. 인덱스 종류 — 무엇을 읽는가 (SummaryIndex · TreeIndex)

| 인덱스 | 질의 시 읽는 범위 | 적합한 작업 |
|--------|------------------|------------|
| `VectorStoreIndex` | 유사 **top-k** 노드 | 특정 질문 **QA** |
| `SummaryIndex` | **모든 노드** | **전체 요약**·전수 검토 |
| `TreeIndex` | 루트→자식 **계층 트리** | 큰 코퍼스 하향 탐색·요약 |

`SummaryIndex`/`TreeIndex` 를 같은 코퍼스로 만들어 본다. (TreeIndex 는 노드 그룹을 LLM이
요약해 트리를 구성한다.)

In [5]:
summary_index = SummaryIndex.from_documents(DOCS)   # 모든 노드 보유(요약형)
tree_index    = TreeIndex.from_documents(DOCS)      # 계층 트리 구성(LLM 요약)
print("SummaryIndex 노드수:", len(summary_index.docstore.docs), "(전체 보유)")
print("TreeIndex    노드수:", len(tree_index.docstore.docs))

# TreeIndex 질의 — 트리를 따라 내려가며 답한다
tr = tree_index.as_query_engine().query(EVAL_QUESTION)
print("\n[TreeIndex] 답변:", str(tr).strip()[:160])

SummaryIndex 노드수: 10 (전체 보유)
TreeIndex    노드수: 10

[TreeIndex] 답변: RAG와 파인튜닝의 핵심 차이는 **무엇을 해결하느냐**입니다.

- **파인튜닝**
  - 사전학습된 모델의 가중치를 도메인 데이터로 추가 학습하는 방식
  - **말투, 형식, 특정 작업 성능**을 높이는 데 강함
  - **행동이나 스타일을 고정**하고 싶을 때 유리함
  - 반면


## 4. SummaryIndex + tree_summarize — "전체 문서를 요약해줘"

검색형(top-k)은 일부만 보지만, **`SummaryIndex` + `response_mode="tree_summarize"`** 는
**모든 노드를 읽어** 부분 요약을 트리로 합쳐 **코퍼스 전체 요약**을 만든다. `source_nodes`
개수가 **전체 문서 수**와 같음을 확인한다(검색형 top-k=3과 대비).

In [6]:
summary_qe = summary_index.as_query_engine(response_mode="tree_summarize")
summ = summary_qe.query("전체 문서를 3문장으로 요약해줘")
print("[SummaryIndex tree_summarize] 전체 요약:")
print(str(summ).strip()[:300])
print(f"\n읽은 source_nodes: {len(summ.source_nodes)}개  (= 전체 문서 {len(DOCS)}개를 모두 읽음)")

# 대비: 검색형(top-k=3)으로 같은 요약을 시키면 3개만 본다
v = qe.query("전체 문서를 3문장으로 요약해줘")
print(f"검색형 VectorStore(top_k=3) source_nodes: {len(v.source_nodes)}개 (일부만 → 전체 요약엔 부적합)")

[SummaryIndex tree_summarize] 전체 요약:
이 문서는 LLM, 트랜스포머, 프롬프트 엔지니어링의 기본 개념과 함께, LLM의 최신 지식 한계와 환각 문제를 설명한다.  
또한 RAG의 동작 방식과 장점, 그리고 이를 구성하는 임베딩, 벡터DB, 청킹의 역할 및 설계 원칙을 정리한다.  
마지막으로 파인튜닝과 RAG의 용도 차이, 에이전트와의 결합 가능성, 그리고 RAG 시스템을 검색 품질과 생성 품질로 평가하는 방법을 소개한다.

읽은 source_nodes: 10개  (= 전체 문서 10개를 모두 읽음)
검색형 VectorStore(top_k=3) source_nodes: 3개 (일부만 → 전체 요약엔 부적합)


**포인트 — 작업에 맞는 인덱스를 고른다**

- **"전체를 요약"** 류 질의는 검색형(top-k)으로는 일부 문서가 빠진다 → **SummaryIndex + tree_summarize**(전수 읽기)
- `tree_summarize` = 노드별 부분 요약을 **트리로 재귀 병합** → 컨텍스트 한도를 넘는 코퍼스도 요약 가능
- **TreeIndex** 는 인덱스 자체를 계층으로 구성(빌드 시 LLM 요약) → 탐색형·대규모에 유리
- 즉 LlamaIndex의 힘은 "한 줄 인덱싱"뿐 아니라 **작업에 맞는 인덱스 추상화 선택**에 있다

# [TODO]

## [TODO] 1. 다른 질문으로 VectorStore QA

`qe.query(...)` 에 **다른 질문**(예: "벡터 데이터베이스에는 어떤 종류가 있나?")을 넣어
답변과 출처를 확인하세요.

In [7]:
# [TODO] 1: 다른 질문으로 query
# [TODO] 여기에 코드를 작성하세요.
v = qe.query("벡터 데이터베이스에는 어떤 종류가 있나?")
print(f"검색형 VectorStore(top_k=3) source_nodes: {len(v.source_nodes)}개 (일부만 → 전체 요약엔 부적합)")

검색형 VectorStore(top_k=3) source_nodes: 3개 (일부만 → 전체 요약엔 부적합)


## [TODO] 2. SummaryIndex + tree_summarize 로 다른 전체 요약

`summary_index.as_query_engine(response_mode="tree_summarize")` 로 **"이 문서들이 다루는
주제(topic)들을 나열해 한 문단으로 요약해줘"** 를 질의하고, 읽은 `source_nodes` 개수가
전체 문서 수와 같은지 확인하세요.

In [ ]:
# [TODO] 2: SummaryIndex tree_summarize 로 전체 주제 요약
# [TODO] 여기에 코드를 작성하세요.
summary_qe = summary_index.as_query_engine(response_mode="tree_summarize")
summ = summary_qe.query("이 문서들이 다루는주제(topic)들을 나열해 한 문단으로 요약해줘")
print("[SummaryIndex tree_summarize] 전체 요약:")
print(str(summ).strip()[:300])
print(f"\n읽은 source_nodes: {len(summ.source_nodes)}개  (= 전체 문서 {len(DOCS)}개를 모두 읽음)")


[SummaryIndex tree_summarize] 전체 요약:
이 문서들은 RAG(검색 증강 생성), 파인튜닝, 임베딩, 벡터DB, 청킹, LLM, 트랜스포머, 프롬프트 엔지니어링, AI 에이전트, 평가를 다룬다. 전체적으로는 대규모 언어 모델 기반 시스템을 구축할 때 필요한 핵심 요소들을 설명하며, 외부 문서 검색을 통한 최신·비공개 지식 활용, 도메인 맞춤 학습, 의미 기반 검색을 위한 벡터 표현과 저장, 문서 분할 전략, 모델의 생성 원리와 한계, 프롬프트 설계, 도구를 활용하는 에이전트 구조, 그리고 검색·생성 품질을 함께 측정하는 평가 방법까지 아우른다.

읽은 source_nodes: 10개  (= 전체 문서 10개를 모두 읽음)
검색형 VectorStore(top_k=3) source_nodes: 3개 (일부만 → 전체 요약엔 부적합)


## 실습 정리

- LlamaIndex = 데이터 인덱싱·검색 특화, `from_documents` 한 줄
- **인덱스 종류로 '무엇을 읽는가'가 달라진다**: VectorStore(top-k 검색)·Summary(전체)·Tree(계층)
- **전체 요약**엔 `SummaryIndex` + `tree_summarize`(전수 읽기·재귀 병합)가 정석
- 같은 코퍼스로 6교시에서 Haystack·DSPy와 비교